## ESAT Uncertainty Perturbation Workflow

This notebook implements an uncertainty perturbation workflow for model evaluation.


#### Code Imports

In [ ]:
import time
import copy

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from tqdm.notebook import trange, tqdm, tnrange
from plotly.subplots import make_subplots
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import min_weight_full_bipartite_matching
from esat.data.datahandler import DataHandler
from esat.model.sa import SA
from esat.model.batch_sa import BatchSA
from esat.data.analysis import ModelAnalysis, BatchAnalysis
from esat.error.bootstrap import Bootstrap
from esat_eval.simulator import Simulator
from esat_eval.factor_comparison import FactorCompare
from esat_eval.factor_catalog import FactorCatalog

pd.options.display.float_format = '{:.4f}'.format

In [ ]:
# Synethic dataset parameter value ranges
syn_factors_min = 3
syn_factors_max = 8

syn_features_min = 15
syn_features_max = 45

syn_samples_min = 200
syn_samples_max = 1000

outliers = True
outliers_p_min = 0.05
outliers_p_max = 0.1
outliers_mag_min = 1.1
outliers_mag_max = 2

noise_mean_min = 0.05
noise_mean_max = 0.15
noise_scale = 0.01

uncertainty_mean_min = 0.05
uncertainty_mean_max = 0.15
uncertainty_scale = 0.01

contr_curve_min_range = [0.0, 1.0]
contr_curve_max_range = [2.0, 5.0]
contr_curve_scale_range = [0.1, 0.5]

random_seed = 337
k_coef = 0.75

In [ ]:
rng = np.random.default_rng(seed=random_seed)

In [ ]:
# Initialize the simulator with the above parameters
def generate_synthetic_data(true_factor):
    n_features = rng.integers(low=syn_features_min, high=syn_features_max, size=1)[0]
    n_samples = rng.integers(low=syn_samples_min, high=syn_samples_max, size=1)[0]
    i_outlier_p = round(rng.uniform(low=outliers_p_min, high=outliers_p_max, size=1)[0], 2)
    i_outlier_mag = round(rng.uniform(low=outliers_mag_min, high=outliers_mag_max, size=1)[0], 2)
    contribution_max = round(rng.uniform(low=1.0, high=10.0, size=1)[0], 2)
    print(f"True Factors: {true_factor}, Features: {n_features}, Samples: {n_samples}, Outliers %: {i_outlier_p}, Outliers Magnitude: {i_outlier_mag}, Contribution Max: {contribution_max}")
    simulator = Simulator(seed=rng.integers(low=0, high=10, size=1)[0],
                          factors_n=true_factor,
                          features_n=n_features,
                          samples_n=n_samples,
                          outliers=outliers,
                          outlier_p=i_outlier_p,
                          outlier_mag=i_outlier_mag,
                          contribution_max=contribution_max,
                          noise_mean_min=noise_mean_min,
                          noise_mean_max=noise_mean_max,
                          noise_scale=noise_scale,
                          uncertainty_mean_min=uncertainty_mean_min,
                          uncertainty_mean_max=uncertainty_mean_max,
                          uncertainty_scale=uncertainty_scale,
                          verbose=False
                         )
    curved_factors_count = rng.integers(low=0, high=true_factor, size=1)[0]
    curved_factor_list = rng.choice(list(range(true_factor)), size=curved_factors_count, replace=False)
    for c_i in curved_factor_list:
        # parameters not used by the curve type are ignored
        i_curve_type = rng.choice(['uniform', 'decreasing', 'increasing', 'logistic', 'periodic'], size=1)[0]
        i_curve_min = rng.uniform(low=contr_curve_min_range[0], high=contr_curve_min_range[1], size=1)[0]
        i_curve_max = rng.uniform(low=contr_curve_max_range[0], high=contr_curve_max_range[1], size=1)[0]
        i_curve_scale = rng.uniform(low=contr_curve_scale_range[0], high=contr_curve_scale_range[1], size=1)[0]
        i_curve_frequency = rng.uniform(low=0.1, high=0.9, size=1)[0]
        
        # To keep all as uniform comment out the line below
        # simulator.update_contribution(factor_i=c_i, curve_type=i_curve_type, scale=i_curve_scale, frequency=i_curve_frequency, minimum=i_curve_min, maximum=i_curve_max)
    
    syn_input_df, syn_uncertainty_df = simulator.get_data()
    data_handler = DataHandler.load_dataframe(input_df=syn_input_df, uncertainty_df=syn_uncertainty_df)
    data_handler.metrics
    V, U = data_handler.get_data()
    return V, U

# # Optimized Factor Search Algorithm
# # Step 1: check indices of max/min values (corr,emc/raae) by column. If all are unique this is the optimal mapping.
# # Step 2: perform a min weight full bipartite matching, where each column index is mapped to a row index to maximize or minimize the sum of matching.
# def optimal_mapping_algorithm(batch_results, mapping_type: str = "raae"):
#     n_factors = batch_results["k"]
#     batch_mapping = {}
#     batch_values = {}
#     for i, p_model in batch_results["perturb results"].items():
#         model_mapping = None
#         mapping_values = [-1 for i in range(n_factors)]
#         mapping_matrix = None
#         if mapping_type == "raae":
#             optimal_indices = np.array(p_model["all_raae"]).argmin(axis=0)
#             mapping_matrix = np.array(p_model["all_raae"])
#             maximize = False
#         elif mapping_type == "emc":
#             optimal_indices = np.array(p_model["all_emc"]).argmax(axis=0)
#             mapping_matrix = np.array(p_model["all_emc"])
#             maximize = True
#         else:
#             optimal_indices = np.array(p_model["all_corr"]).argmax(axis=0)
#             mapping_matrix = np.array(p_model["all_corr"])
#             maximize = True

#         # Step 1, all optimal value indices are unique and no other values need to be checked.
#         if (np.unique(optimal_indices, return_counts=True)[1].max() == 1):
#             model_mapping = optimal_indices
#         else:
#             m_bi_matrix = csr_matrix(mapping_matrix)
#             model_mapping = list(min_weight_full_bipartite_matching(m_bi_matrix, maximize=maximize))
#         optimal_index_tuples = list(zip(list(range(n_factors)), model_mapping))
#         for j, oi in enumerate(optimal_index_tuples):
#             ele_values = mapping_matrix[oi]
#             mapping_values[j] = np.round(ele_values, 4)
        
#         batch_mapping[i] = optimal_indices
#         batch_values[i] = mapping_values
        
#     return batch_mapping, batch_values


def perturb_uncertainty(u, _rng, perturb_p = 0.25, sigma = 0.25):
    i_u = copy.copy(u)
    if isinstance(perturb_p, float):
        perturb_p = [perturb_p for i in range(u.shape[1])]
    elif isinstance(perturb_p, list) and len(perturb_p) != u.shape[1]:
        perturb_p = [perturb_p[0] for i in range(u.shape[1])]
    if isinstance(sigma, float):
        sigma = [sigma for i in range(u.shape[1])]
    elif isinstance(sigma, list) and len(sigma) != u.shape[1]:
        sigma = [sigma[0] for i in range(u.shape[1])]
    i_m = np.zeros(shape=i_u.shape)
    for i, _p in enumerate(perturb_p):
        i_mask = _rng.random(size=u[:,i].shape) > _p
        i_mean = 0.0
        i_logn = _rng.lognormal(i_mean, sigma[i], size=u[:,i].shape)
        i_m[:,i] = i_logn
        i_m[:,i][i_mask] = i_m[:,i][i_mask]
        ij_u = i_u[:,i]
        ij_u = ij_u * i_logn
        ij_u[i_mask] = ij_u[i_mask]
        ij_u[ij_u <= 0.0] = 1e-12
        i_u[:,i] = ij_u
    return i_u, i_m


def run_perturbation(v, u, factors, random_seed, u_collection = None, perturb_p = 1.0, sigma = 0.33, sa_model=None, models=10, max_iter=10000, converge_n=50, converge_delta=0.01, threshold: float=0.9, pg_leave=True, verbose=False):
    # Runs a perturbation input batch instance
    # Steps:
    # 1. Create a SA instance using the provided iV, iU and true_k for the data and factor count (if one is not provided).
    # 2. Using a lognormal std (sigma) and % instance change (perturb_p) for the uncertainty:
    #    a. Select perturb_p number of indecies from the uncertainty data matrix and change those values by a random selection from a lognormal distribution with mean=1 and sigma/std=the provided valude (default=0.25)
    #    b. Use a provided collection of pre-defiend uncertainty matrices
    # 3. With the perturbed uncertainty rerun the model using the base model H matrix.
    # 4. Repeat for n number of models
    # 5. Evaluate the results from all the perturb model profiles and concentrations, the ones that mapped (had a correlation above the threhsold) and provide the range of values for the factors.
    rng = np.random.default_rng(seed=random_seed)
    # step 1
    if sa_model is None:
        sa_model = SA(V=v, U=u, factors=factors, seed=random_seed, verbose=verbose, method="ls-nmf")
        sa_model.initialize()
        sa_model.train(max_iter=max_iter, converge_delta=converge_delta, converge_n=converge_n)

    # base_mean_W = np.mean(sa_model.W, axis=0)[0]
    # base_mass_matrix = (base_mean_W*sa_model.H)/np.sum(base_mean_W*sa_model.H)    # base mass matrix
    
    # perturb_results = {
    #     "k": factors,
    #     "seed": random_seed,
    #     "base model": sa_model,
    #     "perturb %": perturb_p,
    #     "perturb sigma": sigma,
    #     "perturb results": {}
    # }

    #TODO: Parallelize when added to the code base
    perturbed_models = []
    for i in tnrange(models, desc="Running Perturbations on base model", leave=pg_leave):
        if u_collection is None:
            i_u, i_m = perturb_uncertainty(u=u, perturb_p=perturb_p, sigma=sigma, _rng=rng)
        elif i > len(u_collection):
            i_u, i_m = perturb_uncertainty(u=u, perturb_p=perturb_p, sigma=sigma, _rng=rng)
        else:
            i_u = u_collection[i]
            i_m = None
            
        i_sa_model = SA(v, U=i_u, factors=factors, seed=random_seed, verbose=verbose, method="ls-nmf")
        i_sa_model.initialize(H=sa_model.H, W=sa_model.W)
        i_sa_model.train(max_iter=max_iter, converge_delta=converge_delta, converge_n=converge_n)
        perturbed_models.append(i_sa_model)
        
        # i_results = {
        #     "model": copy.copy(i_sa_model),
        #     "multiplier": i_m,
        #     "corr": [],
        #     "corr_mapping": [],
        #     "all_corr": [],
        #     "raae": [],
        #     "raae_mapping": [],
        #     "all_raae": [],
        #     "emc": [],
        #     "emc_mapping": [],
        #     "all_emc": []
        # }
        # n = 1/sa_model.W[:,0].shape[0]

        # i_W_mean = np.mean(i_sa_model.W, axis=0)[0]
        # i_mass_matrix = (i_W_mean*i_sa_model.H)/np.sum(i_W_mean*i_sa_model.H)   # perturbed mass matrix
        
        # for j in range(factors):
        #     j_W = sa_model.W[:,j]      # Base model W column j (factor contribution)
        #     j_H = sa_model.H[j]        # Base model H row j (factor profile)

        #     # Tacking results of equation 4
        #     j_r2 = 0.0
        #     r2_best = -1
        #     all_corr = []

        #     # Tracking results of equation 5
        #     j_raae = float("inf")
        #     raae_best = -1
        #     all_raae = []

        #     # Tracking results of equation 7
        #     j_emc = 0.0
        #     best_emc = -1
        #     all_emc = []
            
        #     for k in range(factors):
        #         k_W = i_sa_model.W[:,k]    # Perturbed model i, W column j (perturbed factor contribution j)
        #         jk_r2 = FactorCompare.calculate_correlation(factor1=j_W.flatten(), factor2=k_W.flatten())    # Equation 4
        #         jk_raae = (np.sum(np.abs(k_W - j_W))*n)/(np.sum(j_W)*n)                                      # Equation 5
        #         jk_emc = FactorCompare.calculate_correlation(factor1=base_mass_matrix[j], factor2= i_mass_matrix[k]) # Equation 7
                
        #         if jk_r2 > j_r2:
        #             r2_best = k 
        #             j_r2 = jk_r2
        #         if jk_raae < j_raae:
        #             j_raae = jk_raae
        #             raae_best = k
        #         if jk_emc > j_emc:
        #             j_emc = jk_emc
        #             best_emc = k
        #         all_corr.append(jk_r2)
        #         all_raae.append(jk_raae)
        #         all_emc.append(jk_emc)
        #     i_results["corr"].append(j_r2)
        #     i_results["corr_mapping"].append(r2_best)
        #     i_results["all_corr"].append(all_corr)
        #     i_results["raae"].append(j_raae)
        #     i_results["raae_mapping"].append(raae_best)
        #     i_results["all_raae"].append(all_raae)
        #     i_results["emc"].append(j_emc)
        #     i_results["emc_mapping"].append(best_emc)
        #     i_results["all_emc"].append(all_emc)
        # perturb_results["perturb results"][i] = i_results
    return sa_model, perturbed_models

In [ ]:
from tqdm import notebook as tqdm_nb

class FactorCompareV2:
    """
    Comparing factors between a base model and a list of other models, providing a mapping between the base model and
    each model in the models list.

    Parameters
    ----------
    base_model : BaseModel
        The base model to compare against.
    models : list
        A list of models to compare against the base model.

    """
    def __init__(self, base_model: SA, models: list, in_notebook: bool = False):
        self.base_model = base_model
        self.models = models
        self.factors = base_model.factors

        self.correlation_data = {}
        self.factor_map = None

        self.calculate_correlation_matrix(in_notebook=in_notebook)

    def calculate_correlation_matrix(self, in_notebook: bool = False):
        """
        Correlation matrices are calculated between a reference base model and a collection of separate models
        (independent or perturbed). The correlation metrics used for comparison are implemented as defined in the
        publication https://doi.org/10.1021/es800085t.

        Parameters
        ----------
        in_notebook : bool
            If True, the function will display a progress bar formatted for Jupyter notebooks.

        Returns
        -------

        """
        base_mean_W = np.mean(self.base_model.W, axis=0)[0]
        base_mass_matrix = (base_mean_W*self.base_model.H)/np.sum(base_mean_W*self.base_model.H)
        n = 1/self.base_model.W[:,0].shape[0]
        
        for i in tqdm_nb.trange(int(len(self.models)), desc="Calculating correlation between base and model factors", leave=False):
            i_results = {
                "corr": [],
                "corr_mapping": [],
                "all_corr": [],
                "raae": [],
                "raae_mapping": [],
                "all_raae": [],
                "emc": [],
                "emc_mapping": [],
                "all_emc": []
            }

            i_model = self.models[i]
            i_mean_W = np.mean(i_model.W, axis=0)[0]
            i_mass_matrix = (i_mean_W*i_model.H)/np.sum(i_mean_W*i_model.H)

            for j in range(self.factors):
                j_W = self.base_model.W[:, j]  # Base model W column j (factor contribution)
                j_H = self.base_model.H[j]  # Base model H row j (factor profile)

                # Tacking results of equation 4
                j_r2 = 0.0
                r2_best = -1
                all_corr = []

                # Tracking results of equation 5
                j_raae = float("inf")
                raae_best = -1
                all_raae = []

                # Tracking results of equation 7
                j_emc = 0.0
                best_emc = -1
                all_emc = []

                for k in range(self.factors):
                    k_W = i_model.W[:, k]  # Perturbed model i, W column j (perturbed factor contribution j)
                    jk_r2 = FactorCompare.calculate_correlation(factor1=j_W.flatten(),
                                                                factor2=k_W.flatten())  # Equation 4
                    jk_raae = (np.sum(np.abs(k_W - j_W)) * n) / (np.sum(j_W) * n)  # Equation 5
                    jk_emc = FactorCompare.calculate_correlation(factor1=base_mass_matrix[j],
                                                                 factor2=i_mass_matrix[k])  # Equation 7

                    if jk_r2 > j_r2:
                        r2_best = k
                        j_r2 = jk_r2
                    if jk_raae < j_raae:
                        j_raae = jk_raae
                        raae_best = k
                    if jk_emc > j_emc:
                        j_emc = jk_emc
                        best_emc = k
                    all_corr.append(jk_r2)
                    all_raae.append(jk_raae)
                    all_emc.append(jk_emc)
                i_results["corr"].append(j_r2)
                i_results["corr_mapping"].append(r2_best)
                i_results["all_corr"].append(all_corr)
                i_results["raae"].append(j_raae)
                i_results["raae_mapping"].append(raae_best)
                i_results["all_raae"].append(all_raae)
                i_results["emc"].append(j_emc)
                i_results["emc_mapping"].append(best_emc)
                i_results["all_emc"].append(all_emc)
            self.correlation_data[i] = i_results

    def determine_map(self, method: str="raae"):
        """
        Determine the factor mapping between the base model and a collection of models.

        Parameters
        ----------
        method : str
            Correlation method to use, options include: "corr", "raae", "emc".
        """
        batch_mapping = {}
        batch_values = {}
        for i, p_model in self.correlation_data.items():
            mapping_values = [-1 for i in range(self.factors)]
            if method == "raae":
                optimal_indices = np.array(p_model["all_raae"]).argmin(axis=0)
                mapping_matrix = np.array(p_model["all_raae"])
                maximize = False
            elif method == "emc":
                optimal_indices = np.array(p_model["all_emc"]).argmax(axis=0)
                mapping_matrix = np.array(p_model["all_emc"])
                maximize = True
            else:
                optimal_indices = np.array(p_model["all_corr"]).argmax(axis=0)
                mapping_matrix = np.array(p_model["all_corr"])
                maximize = True

            # Step 1, all optimal value indices are unique and no other values need to be checked.
            if (np.unique(optimal_indices, return_counts=True)[1].max() == 1):
                model_mapping = optimal_indices
            else:
                m_bi_matrix = csr_matrix(mapping_matrix)
                model_mapping = list(min_weight_full_bipartite_matching(m_bi_matrix, maximize=maximize))
            optimal_index_tuples = list(zip(list(range(self.factors)), model_mapping))
            for j, oi in enumerate(optimal_index_tuples):
                ele_values = mapping_matrix[oi]
                mapping_values[j] = np.round(ele_values, 4)

            batch_mapping[i] = optimal_indices
            batch_values[i] = mapping_values
        return batch_mapping, batch_values


## Single Perturbation Instance

A single instance of the run_perturbation function is called. This will create a single SA instance to use as the base model with n_models being the number of perturbed instances to make (each independent of each other) for that single base model.

In [ ]:
%%time
true_k = 6
iV, iU = generate_synthetic_data(true_factor=true_k)

In [ ]:
estimate_k = 6
n_models = 50
threshold = 0.9

perturb_p = 1.0
perturb_sigma = 0.25

base_model, perturbed_models = run_perturbation(factors=estimate_k, v=iV, u=iU, random_seed=random_seed, perturb_p=perturb_p, sigma=perturb_sigma, models=n_models, max_iter=20000, converge_n=20, converge_delta=0.01, threshold=threshold)

In [ ]:
fc = FactorCompareV2(base_model=base_model, models=perturbed_models, in_notebook=True)

In [ ]:
perturb_mapping, perturb_values = fc.determine_map(method="raae")
perturb_values

In [ ]:
# Metrics
# Factor mapping (what percentage of perturbed model factors map to the base model factors, using corr(W), RAAE or EMC)
# Factor profile ranges (among mapped perturbed model factors)
# Factor mean contribution ranges (among mapped pertrubed model factors)
n_factors = estimate_k
corr_mapping = [0] * n_factors
raae_mapping = [0] * n_factors
emc_mapping = [0] * n_factors
for i, p_model in batch_results0["perturb results"].items():
    cmap_dif = np.subtract(p_model["corr_mapping"], list(range(n_factors)))
    raae_dif = np.subtract(p_model["raae_mapping"], list(range(n_factors)))
    emc_dif = np.subtract(p_model["emc_mapping"], list(range(n_factors)))
    for j in range(n_factors):
        if cmap_dif[j] == 0:
            corr_mapping[j] += 1
        if raae_dif[j] == 0:
            raae_mapping[j] += 1
        if emc_dif[j] == 0:
            emc_mapping[j] += 1
mapping_df = pd.DataFrame(data=
                          {
                              "Factor": list(range(n_factors)), 
                              "R2": np.round(100 * np.array(corr_mapping)/n_models, 2), 
                              "RAAE": np.round(100 * np.array(raae_mapping)/n_models, 2), 
                              "EMC": np.round(100 * np.array(emc_mapping)/n_models, 2)
                          }
                         )
mapping_df

In [ ]:
df_columns = [f"Factor {i}" for i in range(n_factors)]
df_index = [f"Base Factor {i}" for i in range(n_factors)]
corr_df = pd.DataFrame(data=np.array(batch_results0["perturb results"][0]["all_corr"]), columns=df_columns, index=df_index)
print("Correlation Matrix")
corr_df

In [ ]:
raae_df = pd.DataFrame(data=np.array(batch_results0["perturb results"][0]["all_raae"]), columns=df_columns, index=df_index)
print("RAAE Matrix")
raae_df

In [ ]:
emc_df = pd.DataFrame(data=np.array(batch_results0["perturb results"][1]["all_emc"]), columns=df_columns, index=df_index)
print("EMC Matrix")
emc_df

In [ ]:
emc_mapping_results = optimal_mapping_algorithm(batch_results=batch_results0, mapping_type="emc")
emc_mapping_results

In [ ]:
base_model = batch_results0['base model']

base_model.W[:,0].shape

In [ ]:
# Table 1
u_m_stats = np.percentile(batch_results0["perturb results"][0]["multiplier"], [1, 25, 50, 75, 99])
m_stats_dict = dict(zip(["first", "25th", "50th", "75th", "99th"], u_m_stats))
m_stats_dict["sigma"] = perturb_sigma
m_stats_df = pd.Series(data=m_stats_dict)
m_stats_df

In [ ]:
base_H = batch_results0["base model"].H
base_H = (base_H / np.sum(base_H, axis=0))

base_W = batch_results0["base model"].W
base_W = (base_W.T / np.sum(base_W.T, axis=0)).T

threshold = 0.95

p_Hs = []
p_Ws = []
p_Qs = []
for p, p_results in batch_results0["perturb results"].items():
    p_model = p_results["model"]
    p_H = (p_model.H / np.sum(p_model.H, axis=0))
    p_W = (p_model.W.T / np.sum(p_model.W.T, axis=0)).T
    _p_H = []
    _p_W = []
    p_mapping = emc_mapping_results[0][p]
    p_map_metric = np.array(emc_mapping_results[1][p]) > threshold
    for i in range(p_model.factors):
        if p_map_metric[i]:
            _p_H.append(p_H[p_mapping[i]])
            _p_W.append(p_W[:,p_mapping[i]])
        # else:
        #     _p_H.append(base_H[p_mapping[i]])
    _p_H = np.array(_p_H)
    _p_W = np.array(_p_W)
    p_Hs.append(_p_H)
    p_Ws.append(_p_W)
    p_Qs.append(p_model.Qtrue)
perturb_H = np.dstack(p_Hs)
perturb_W = np.dstack(p_Ws)
mean_perturb_H = np.mean(perturb_H, axis=2)
std_perturb_H = np.std(perturb_H, axis=2)
min_perturb_H = np.min(perturb_H, axis=2)
max_perturb_H = np.max(perturb_H, axis=2)

feature_labels = [f"Feature {i+1}" for i in range(base_H.shape[1])]
print(f"QTrue - Base: {np.round(batch_results0["base model"].Qtrue,4)}, Perturb Mean: {np.round(np.mean(p_Qs),4)}, Perturb STD: {np.round(np.std(p_Qs),4)}, Perturb Min: {np.round(np.min(p_Qs),4)}, Perturb Max: {np.round(np.max(p_Qs),4)}")

In [ ]:
factor_i = 1
f1_dict = {
    "Base": base_H[factor_i],
    "Mean Perturb": mean_perturb_H[factor_i],
    "% diff": np.round(100*(mean_perturb_H[factor_i] - base_H[factor_i])/((mean_perturb_H[factor_i] + base_H[factor_i])/2) , 4),
    "STD Perturb": std_perturb_H[factor_i],
    "Min Perturb": min_perturb_H[factor_i],
    "Max Perturb": max_perturb_H[factor_i],
}
f1_dict["% diff"][f1_dict["Mean Perturb"] < 1e-6] = 0.0 

f1_df = pd.DataFrame(f1_dict, index=feature_labels).round(10)
f1_df

In [ ]:
# Figure 1
factor_labels = [f"Factor {i+1}" for i in range(base_W.shape[1])]
p_W_mean = np.mean(perturb_W, axis=(1,2))
base_W_mean = np.mean(base_W, axis=0)
factor_w0_fig = make_subplots()
factor_w0_fig.add_trace(go.Bar(x=factor_labels, y=100*p_W_mean, name="Perturbed W"))
factor_w0_fig.add_trace(go.Bar(x=factor_labels, y=100*base_W_mean, name="Base W"))
factor_w0_fig.update_layout(title=f"Perturbed Mean Factor Contribution Results", width=1200, height=800, hovermode='x unified', barmode='group')
factor_w0_fig.update_yaxes(title_text="Mean Normalized Contributions (%)")
factor_w0_fig.show()

In [ ]:
# Figure 2
factor_w1_fig = make_subplots()
p_W_means = np.mean(perturb_W, axis=1)
base_W_means = np.mean(base_W, axis=0)
for i in range(len(factor_labels)):
    i_W_means = p_W_means[i]
    factor_w1_fig.add_trace(go.Box(y=i_W_means, name=factor_labels[i]))
factor_w1_fig.add_trace(go.Scatter(x=factor_labels, y=base_W_means, name="Base", mode="markers", marker_color="black"))
factor_w1_fig.update_layout(title=f"Perturbed Average Source Contributions - Sigma: {perturb_sigma}", width=1200, height=800, hovermode='x unified')
factor_w1_fig.update_yaxes(title_text="Source Contributions")
factor_w1_fig.show()

In [ ]:
raae_mapping_results = optimal_mapping_algorithm(batch_results=batch_results0, mapping_type="raae")
raae_mapping_results

In [ ]:
# Figure 2
factor_raae_fig = make_subplots()
factor_raae = np.zeros(shape=(n_models, len(factor_labels)))
for i, v in raae_mapping_results[1].items():
    i_mapping = raae_mapping_results[0][i]
    factor_raae[i] = np.array(v)[i_mapping]
for i in range(len(factor_labels)):
    i_raae_means = factor_raae[:,i]
    factor_raae_fig.add_trace(go.Box(y=i_raae_means, name=factor_labels[i]))
factor_raae_fig.add_hline(y=np.mean(factor_raae))
factor_raae_fig.update_layout(title=f"Perturbed Source RAAE - Sigma: {perturb_sigma}", width=1200, height=800, hovermode='x unified')
factor_raae_fig.update_yaxes(title_text="RAAE")
factor_raae_fig.show()

In [ ]:
# Figure 4
factor_emc_fig = make_subplots()
factor_emc = np.zeros(shape=(n_models, len(factor_labels)))
for i, v in emc_mapping_results[1].items():
    i_mapping = emc_mapping_results[0][i]
    factor_emc[i] = np.array(v)[i_mapping]
for i in range(len(factor_labels)):
    i_emc_means = factor_emc[:,i]
    factor_emc_fig.add_trace(go.Box(y=i_emc_means, name=factor_labels[i]))
factor_emc_fig.add_hline(y=np.mean(factor_emc))
factor_emc_fig.update_layout(title=f"Perturbed Source EMC - Sigma: {perturb_sigma}", width=1200, height=800, hovermode='x unified')
factor_emc_fig.update_yaxes(title_text="EMC")
factor_emc_fig.show()

In [ ]:
factor_i = 1
factor_p0_fig = make_subplots(specs=[[{"secondary_y": True}]])
for i in range(len(feature_labels)):
    f_feature_i = perturb_H[factor_i,i]
    factor_p0_fig.add_trace(go.Box(y=f_feature_i, name=feature_labels[i]), secondary_y=False)
factor_p0_fig.add_trace(go.Scatter(x=feature_labels, y=base_H[factor_i], name="Base", mode="markers", marker_color="black"), secondary_y=False)
factor_p0_fig.update_layout(title=f"Perturbed Factor {factor_i} Profile Results", width=1200, height=800, hovermode='x unified')
factor_p0_fig.update_yaxes(title_text="Normalized Profile", secondary_y=False)
factor_p0_fig.show()